In [ ]:
# Core
import os
import pandas as pd
import copy
import numpy as np
import pybedtools

# Graphs
import matplotlib.pyplot as plt
import seaborn as sns
# Statistics
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import binomtest
from adjustText import adjust_text
from sklearn.metrics import precision_recall_curve
from scipy.stats import spearmanr
from scipy.stats import ks_2samp # Kolmogorov–Smirnov (KS)

# --- Base directory (EDIT THIS) ---
# Paths below are relative to this lab-data root (the Collaboration directory).
BASE_DIR = '/home/labs/davidgo/Collaboration'   # <- set to your local copy
os.chdir(BASE_DIR)

In [ ]:
lineage = 'human' # modern or archaic or human
cell_type = 'chondrocytes' #osteoblasts or NPCs
filter_ATAC_seq = False
filter_SCREEN = False
filter_1_var_per_oligo = True
data_type = 'FIMO' # PBM or FIMO

print(f'lineage: {lineage}')
print(f'cell type: {cell_type}') 
print(f'filtering by ATAC-seq: {filter_ATAC_seq}')
print(f'filtering by SCREEN: {filter_SCREEN}')
print(f'data type: {data_type}')
#print(f'filtering for oligos with a single variant: {filter_1_var_per_oligo}')

# Process archaic osteoblasts and NPCs

In [ ]:
print(data_type)
if (lineage == 'human') & (cell_type == 'chondrocytes') & (data_type == 'PBM'):
    TF_data = pd.read_csv(f'humanMPRA/TF_analysis/final/variant_tool_outputs/{data_type}_diff_binding_scores.tsv', sep = '\t',
                         header=0)
    
if (lineage == 'human') & (cell_type == 'chondrocytes') & (data_type == 'FIMO'):
    TF_data = pd.read_csv(f'humanMPRA/TF_analysis/final/variant_tool_outputs/{data_type}_diff_binding_scores.tsv', sep = '\t',
                         header=0)



In [ ]:
TF_data_processed = TF_data.copy()
TF_name_long = TF_data_processed['motif_id:TF:start_motif-stop_motif:strand'].str.split(':')
if data_type == 'FIMO':
    TF_data_processed['motif_id'] = TF_name_long.str[1]
elif data_type == 'PBM':
    TF_data_processed['motif_id'] = TF_name_long.str[0]

TF_data_processed['Mus_musculus'] = TF_data_processed['motif_id'].str.contains('_mus_musculus', regex=False).fillna(False)
TF_data_processed['motif_id'] = TF_data_processed['motif_id'].str.replace(r'_mus_musculus.*$', '', regex=True)
TF_data_processed['motif_id_clean'] = TF_data_processed['motif_id'].str.split('_').str[0]

### If there are multiple motifs for the same variant, keep the one with the highest absolute z-score

In [ ]:
TF_data_processed = TF_data_processed.loc[
    TF_data_processed['diff_binding_zscore'].abs().sort_values(ascending=False).index
].drop_duplicates(
    subset=['chr', 'start', 'stop', 'motif_id_clean'], keep='first'
).reset_index(drop=True)

### Total number of unique TFs:

In [ ]:
TFs = TF_data_processed['motif_id_clean'].unique()
print(f'number of unique TFs: {len(TFs)}')

# merge with oligo data 

In [ ]:

raw_data = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     header=0)

min_DNA_counts = 50

raw_data['minimum_DNA_counts'] = (
    raw_data[['DNA_counts_raw_ancestral', 'DNA_counts_raw_derived']]
    .min(axis=1)
)
print('before filtering:')
print(len(raw_data))
raw_data = raw_data[raw_data["minimum_DNA_counts"] >= min_DNA_counts]
print('after filtering:')
print(len(raw_data))


### Add activity and diff activity bool cols

In [ ]:
raw_data['diff_active_bool'] = raw_data['differential_activity']==True
raw_data['active_bool'] = raw_data['differential_activity'].isin([True, False])

### explode oligos based on variants (so that each varaint gets its own row)

In [ ]:
raw_data['variants_genomic_og'] = raw_data['variants_genomic'].copy()

raw_data = (
    raw_data.assign(
        variants_genomic=raw_data['variants_genomic'].fillna('').str.split(';')
    )
    .explode('variants_genomic')
)

raw_data['variants_genomic'] = raw_data['variants_genomic'].str.strip()
raw_data = raw_data[raw_data['variants_genomic'].ne('')].reset_index(drop=True)

print(f"Exploded rows: {len(raw_data):,}")
raw_data[['oligo', 'variants_genomic']].head()


### Parse the variants_genomic col

In [ ]:
oligos = copy.deepcopy(raw_data)

# Parse strings like: chr8:30170539(T->C)
parsed = oligos['variants_genomic'].str.extract(r'^(chr[^:]+):(\d+)\(([^)]+)\)$')

oligos['chr'] = parsed[0]
oligos['stop'] = pd.to_numeric(parsed[1], errors='coerce')
oligos['start'] = oligos['stop'] - 1
oligos['mutation'] = parsed[2]

# Optional: drop rows that failed parsing
print(f"Rows before parsing: {len(oligos):,}")
oligos = oligos.dropna(subset=['chr', 'start', 'stop', 'mutation']).copy()
oligos['start'] = oligos['start'].astype(int)
oligos['stop'] = oligos['stop'].astype(int)
print(f"Rows after parsing (should be equal to above): {len(oligos):,}")

### Add metadata to oligos data

In [ ]:
oligos['number_of_variants'] = oligos['variants_genomic_og'].str.count(';') + 1
oligos['motif_association_method'] = data_type
oligos['cell_type_used_for_motif_filtering'] = cell_type

oligos.rename(columns={'oligo':'seq_id',}, inplace=True)

### Merge the TF dataset and the hMPRA dataset based on the variant position

In [ ]:
# Merge oligos with TF_data_processed on 'chr', 'start', 'stop'
oligo_TF_merged = pd.merge(
    oligos,
    TF_data_processed,
    how='inner',
    left_on=['chr', 'start', 'stop'],
    right_on=['chr', 'start', 'stop']
)


## Save output

In [ ]:
if (lineage == 'human') & (cell_type == 'chondrocytes'):
    oligo_TF_merged[['seq_id', 'chr','number_of_variants',
                     'diff_binding_score','diff_binding_zscore','diff_binding_pval','diff_binding_fdr','motif_association_method','cell_type_used_for_motif_filtering',
                     'Mus_musculus','motif_id_clean']].to_csv(f'humanMPRA/TF_analysis/final/results/hMPRA_{data_type}_oligo_TF_merged.tsv', sep='\t', index=False)